## 02: GIN Baseline

Builds a GIN model for ogbg-molhiv binary classification, using OGB's official scaffold split and a class-weighted BCEWithLogitsLoss to handle the label imbalance. Runs a small hyperparameter search over depth, hidden dimension, and dropout, tracks validation ROC-AUC each epoch, and checkpoints the best-performing configuration. Saves the best model's weights to models/gin_best.pt and its test metrics into outputs/results.json.

In [1]:
# from google.colab import drive
# drive.mount('/content/drive')
# BASE_DIR = "/content/drive/MyDrive/molhiv-gnn"

# !pip install torch-geometric ogb rdkit

BASE_DIR = ".."

In [2]:
import pandas as pd
import os
import json
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINConv, global_mean_pool
from ogb.graphproppred import PygGraphPropPredDataset, Evaluator

torch.serialization.add_safe_globals([
    torch_geometric.data.data.DataEdgeAttr,
    torch_geometric.data.data.DataTensorAttr,
    torch_geometric.data.storage.GlobalStorage,
])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)
device

device(type='cuda')

In [3]:
dataset = PygGraphPropPredDataset(name="ogbg-molhiv", root=f"{BASE_DIR}/dataset")
split_idx = dataset.get_idx_split()
evaluator = Evaluator(name="ogbg-molhiv")

train_loader = DataLoader(dataset[split_idx["train"]], batch_size=128, shuffle=True)
valid_loader = DataLoader(dataset[split_idx["valid"]], batch_size=256, shuffle=False)
test_loader = DataLoader(dataset[split_idx["test"]], batch_size=256, shuffle=False)

train_labels = torch.cat([dataset[i].y for i in split_idx["train"]]).squeeze()
n_pos = (train_labels == 1).sum().item()
n_neg = (train_labels == 0).sum().item()
pos_weight = torch.tensor([n_neg / n_pos], device=device)
n_pos, n_neg, pos_weight

(1232, 31669, tensor([25.7054], device='cuda:0'))

In [4]:
from ogb.graphproppred.mol_encoder import AtomEncoder

class GIN(nn.Module):
    def __init__(self, hidden_dim, num_layers, dropout):
        super().__init__()
        self.atom_encoder = AtomEncoder(hidden_dim)
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        for _ in range(num_layers):
            mlp = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
            )
            self.convs.append(GINConv(mlp))
            self.bns.append(nn.BatchNorm1d(hidden_dim))
        self.dropout = dropout
        self.out = nn.Linear(hidden_dim, 1)

    def forward(self, x, edge_index, batch):
        h = self.atom_encoder(x)
        for conv, bn in zip(self.convs, self.bns):
            h = conv(h, edge_index)
            h = bn(h)
            h = F.relu(h)
            h = F.dropout(h, p=self.dropout, training=self.training)
        h = global_mean_pool(h, batch)
        return self.out(h).squeeze(-1)

In [5]:
def train_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index, batch.batch)
        loss = F.binary_cross_entropy_with_logits(out, batch.y.float().view(-1), pos_weight=pos_weight)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch.num_graphs
    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    y_true, y_pred = [], []
    for batch in loader:
        batch = batch.to(device)
        out = model(batch.x, batch.edge_index, batch.batch)
        y_true.append(batch.y.view(-1, 1).cpu())
        y_pred.append(torch.sigmoid(out).view(-1, 1).cpu())
    y_true = torch.cat(y_true, dim=0).numpy()
    y_pred = torch.cat(y_pred, dim=0).numpy()
    return evaluator.eval({"y_true": y_true, "y_pred": y_pred})["rocauc"]

In [6]:
configs = [
    {"hidden_dim": 128, "num_layers": 3, "dropout": 0.2},
    {"hidden_dim": 128, "num_layers": 5, "dropout": 0.5},
    {"hidden_dim": 256, "num_layers": 3, "dropout": 0.5},
    {"hidden_dim": 256, "num_layers": 5, "dropout": 0.2},
]

MAX_EPOCHS = 100
PATIENCE = 20

search_results = []
for cfg in configs:
    model = GIN(**cfg).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    best_val_auc = 0.0
    best_state = None
    patience_ctr = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss = train_epoch(model, train_loader, optimizer)
        val_auc = evaluate(model, valid_loader)

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_state = copy.deepcopy(model.state_dict())
            patience_ctr = 0
        else:
            patience_ctr += 1

        if epoch % 10 == 0 or patience_ctr == 0:
            print(f"config={cfg}  epoch={epoch:>3}  train_loss={train_loss:.4f}  val_auc={val_auc:.4f}")

        if patience_ctr >= PATIENCE:
            break

    search_results.append({"config": cfg, "best_val_auc": best_val_auc, "state_dict": best_state})
    print(f"--> config {cfg} finished with best val ROC-AUC = {best_val_auc:.4f}\n")

config={'hidden_dim': 128, 'num_layers': 3, 'dropout': 0.2}  epoch=  1  train_loss=1.2249  val_auc=0.6804


config={'hidden_dim': 128, 'num_layers': 3, 'dropout': 0.2}  epoch=  2  train_loss=1.1653  val_auc=0.7536


config={'hidden_dim': 128, 'num_layers': 3, 'dropout': 0.2}  epoch=  9  train_loss=1.0543  val_auc=0.7696


config={'hidden_dim': 128, 'num_layers': 3, 'dropout': 0.2}  epoch= 10  train_loss=1.0364  val_auc=0.7260


config={'hidden_dim': 128, 'num_layers': 3, 'dropout': 0.2}  epoch= 14  train_loss=1.0072  val_auc=0.7835


...

config={'hidden_dim': 256, 'num_layers': 5, 'dropout': 0.2}  epoch= 20  train_loss=1.0130  val_auc=0.7915


config={'hidden_dim': 256, 'num_layers': 5, 'dropout': 0.2}  epoch= 28  train_loss=0.9721  val_auc=0.8082


config={'hidden_dim': 256, 'num_layers': 5, 'dropout': 0.2}  epoch= 30  train_loss=0.9583  val_auc=0.7618


config={'hidden_dim': 256, 'num_layers': 5, 'dropout': 0.2}  epoch= 40  train_loss=0.9171  val_auc=0.7612


--> config {'hidden_dim': 256, 'num_layers': 5, 'dropout': 0.2} finished with best val ROC-AUC = 0.8082



In [7]:
best_result = max(search_results, key=lambda r: r["best_val_auc"])
best_config = best_result["config"]
print("best config:", best_config, "val ROC-AUC:", best_result["best_val_auc"])

best_model = GIN(**best_config).to(device)
best_model.load_state_dict(best_result["state_dict"])

test_auc = evaluate(best_model, test_loader)
print("test ROC-AUC:", test_auc)

best config: {'hidden_dim': 256, 'num_layers': 5, 'dropout': 0.2} val ROC-AUC: 0.8081735008818343


test ROC-AUC: 0.7597539543057998


In [8]:
os.makedirs(f"{BASE_DIR}/models", exist_ok=True)
os.makedirs(f"{BASE_DIR}/outputs", exist_ok=True)

torch.save(best_result["state_dict"], f"{BASE_DIR}/models/gin_best.pt")

results_path = f"{BASE_DIR}/outputs/results.json"
results = json.load(open(results_path)) if os.path.exists(results_path) else {}
results["gin"] = {
    "config": best_config,
    "val_rocauc": best_result["best_val_auc"],
    "test_rocauc": test_auc,
}
json.dump(results, open(results_path, "w"), indent=2)
results

{'gin': {'config': {'hidden_dim': 256, 'num_layers': 5, 'dropout': 0.2},
  'val_rocauc': 0.8081735008818343,
  'test_rocauc': 0.7597539543057998}}